#### Extract digits from right to left

In [2]:
from numpy.ma.core import minimum

num = 1234
temp = num
digits = []

while temp > 0:
    digit = temp % 10
    digits.insert(0, digit)
    temp = temp // 10

print(f"digits: {digits}")

digits: [1, 2, 3, 4]


In [3]:
def extract_digits(n):
    if n == 0:
        return [0]

    digits = []
    # Handle negative numbers
    sign = -1 if n < 0 else 1
    n = abs(n)

    while n > 0:
        digits.append(n % 10)
        n = n // 10

    # Reverse to maintain original order
    return [d * sign if i == 0 and sign == -1 else d for i, d in enumerate(reversed(digits))]

# Example
num = 12345
print(extract_digits(num))          # [1, 2, 3, 4, 5]
print(extract_digits(-9876))        # [-9, 8, 7, 6]

[1, 2, 3, 4, 5]
[-9, 8, 7, 6]


In [5]:
from typing import List

class Solution:
    def minimum_cost(self, cost: List[int]) -> int:
        if not cost:
            return 0

        cost.sort(reverse=True)

        total = 0
        n = len(cost)

        for i in range(n):
            if i % 3 != 2:
                total += cost[i]

        return total

l1 = [2, 3, 4, 5, 6, 7]

obj = Solution()
print(obj.minimum_cost(l1))


20


In [1]:
from typing import List

class Solution:
    def earliestFinishTime(self,
                          landStartTime: List[int],
                          landDuration: List[int],
                          waterStartTime: List[int],
                          waterDuration: List[int]) -> int:

        n = len(landStartTime)
        m = len(waterStartTime)

        if n == 0 or m == 0:
            return -1  # invalid case, though constraints say n,m >=1

        min_finish = float('inf')

        # Try all possible combinations
        for i in range(n):
            for j in range(m):

                # Case 1: Land -> Water
                finish_land = landStartTime[i] + landDuration[i]
                start_water = max(finish_land, waterStartTime[j])
                finish1 = start_water + waterDuration[j]

                # Case 2: Water -> Land
                finish_water = waterStartTime[j] + waterDuration[j]
                start_land = max(finish_water, landStartTime[i])
                finish2 = start_land + landDuration[i]

                # Best for this pair
                best_for_pair = min(finish1, finish2)

                # Track global minimum
                min_finish = min(min_finish, best_for_pair)

        return min_finish

In [1]:
class Solution:
    def earliestFinishTime(self, landStartTime: List[int], landDuration: List[int], waterStartTime: List[int], waterDuration: List[int]) -> int:
        if not landStartTime or not waterStartTime:
            return -1  # though constraints ensure at least 1

        # Compute end times
        endL = [landStartTime[i] + landDuration[i] for i in range(len(landStartTime))]
        endW = [waterStartTime[j] + waterDuration[j] for j in range(len(waterStartTime))]

        min_endL = min(endL)
        min_endW = min(endW)

        # minA: land first, using best land
        minA = float('inf')
        for j in range(len(waterStartTime)):
            val = max(min_endL + waterDuration[j], endW[j])
            minA = min(minA, val)

        # minB: water first, using best water
        minB = float('inf')
        for i in range(len(landStartTime)):
            val = max(min_endW + landDuration[i], endL[i])
            minB = min(minB, val)

        return min(minA, minB)

Efficient approach:

Store all positions of each element in a dictionary.
For a query (l, r, x), find how many indices of x lie in [l, r].
Since positions are sorted, use binary search (bisect_left and bisect_right).

Time Complexity:

Preprocessing: O(n)
Each query: O(log n)
Total: O(n + q log n)

In [2]:
from bisect import bisect_left, bisect_right
from collections import defaultdict

class Solution:
    def freqInRange(self, arr, queries):
        pos = defaultdict(list)

        # Store indices of each value
        for i, val in enumerate(arr):
            pos[val].append(i)

        ans = []

        for l, r, x in queries:
            if x not in pos:
                ans.append(0)
                continue

            indices = pos[x]

            left = bisect_left(indices, l)
            right = bisect_right(indices, r)

            ans.append(right - left)

        return ans

In [1]:
from functools import lru_cache

class Solution:
    def totalWaviness(self, num1: int, num2: int) -> int:

        def solve(n: int) -> int:
            if n < 0:
                return 0

            digits = list(map(int, str(n)))
            m = len(digits)

            @lru_cache(None)
            def dp(pos, tight, started, length, prev2, prev1):
                if pos == m:
                    return (1, 0)  # (count, total_waviness)

                limit = digits[pos] if tight else 9

                total_count = 0
                total_waviness = 0

                for d in range(limit + 1):
                    ntight = tight and (d == limit)

                    if not started and d == 0:
                        cnt, wav = dp(
                            pos + 1,
                            ntight,
                            False,
                            0,
                            10,
                            10
                        )
                        total_count += cnt
                        total_waviness += wav

                    elif not started:
                        cnt, wav = dp(
                            pos + 1,
                            ntight,
                            True,
                            1,
                            10,
                            d
                        )
                        total_count += cnt
                        total_waviness += wav

                    else:
                        if length == 1:
                            cnt, wav = dp(
                                pos + 1,
                                ntight,
                                True,
                                2,
                                prev1,
                                d
                            )
                            total_count += cnt
                            total_waviness += wav

                        else:
                            add = 0
                            if (prev1 > prev2 and prev1 > d) or \
                               (prev1 < prev2 and prev1 < d):
                                add = 1

                            cnt, wav = dp(
                                pos + 1,
                                ntight,
                                True,
                                2,
                                prev1,
                                d
                            )

                            total_count += cnt
                            total_waviness += wav + add * cnt

                return (total_count, total_waviness)

            return dp(0, True, False, 0, 10, 10)[1]

        return solve(num2) - solve(num1 - 1)

In [ ]:
from functools import lru_cache

class Solution:
    def totalWaviness(self, num1: int, num2: int) -> int:

        def solve(n: int) -> int:
            if n < 0:
                return 0

            digits = list(map(int, str(n)))
            m = len(digits)

            @lru_cache(None)
            def dp(pos, tight, started, length, prev2, prev1):
                if pos == m:
                    return (1, 0)  # (count, total_waviness)

                limit = digits[pos] if tight else 9

                total_cnt = 0
                total_wav = 0

                for d in range(limit + 1):
                    ntight = tight and (d == limit)

                    # Still leading zeros
                    if not started and d == 0:
                        cnt, wav = dp(
                            pos + 1,
                            ntight,
                            False,
                            0,
                            10,
                            10
                        )
                        total_cnt += cnt
                        total_wav += wav
                        continue

                    if not started:
                        cnt, wav = dp(
                            pos + 1,
                            ntight,
                            True,
                            1,
                            10,
                            d
                        )
                        total_cnt += cnt
                        total_wav += wav
                    else:
                        contrib = 0

                        if length >= 2:
                            # prev1 is the middle digit
                            if (prev1 > prev2 and prev1 > d) or \
                               (prev1 < prev2 and prev1 < d):
                                contrib = 1

                        cnt, wav = dp(
                            pos + 1,
                            ntight,
                            True,
                            length + 1,
                            prev1,
                            d
                        )

                        total_cnt += cnt
                        total_wav += wav + contrib * cnt

                return (total_cnt, total_wav)

            return dp(0, True, False, 0, 10, 10)[1]

        return solve(num2) - solve(num1 - 1)

In [1]:
class Solution:
    def leftRightDifference(self, nums: List[int]) -> List[int]:
        n = len(nums)
        if n == 0:
            return []

        total = sum(nums)
        left_sum = 0
        answer = []

        for i in range(n):
            right_sum = total - left_sum - nums[i]
            answer.append(abs(left_sum - right_sum))
            left_sum += nums[i]

        return answer

In [1]:
class Solution:
    def reverse(self, x: int) -> int:
        INT_MAX = 2**31 - 1
        INT_MIN = -2**31

        sign = -1 if x < 0 else 1
        x = abs(x)

        rev = 0

        while x:
            digit = x % 10
            x //= 10

            if rev > INT_MAX // 10 or (rev == INT_MAX // 10 and digit > 7):
                return 0

            rev = rev * 10 + digit

        rev *= sign

        return rev if INT_MIN <= rev <= INT_MAX else 0

In [ ]:
# Definition for a binary tree node.
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

class Solution:
    def createBinaryTree(self, descriptions: List[List[int]]) -> Optional[TreeNode]:
        nodes = {}
        children = set()

        for parent, child, isLeft in descriptions:
            if parent not in nodes:
                nodes[parent] = TreeNode(parent)

            if child not in nodes:
                nodes[child] = TreeNode(child)

            if isLeft:
                nodes[parent].left = nodes[child]
            else:
                nodes[parent].right = nodes[child]

            children.add(child)

        for val in nodes:
            if val not in children:
                return nodes[val]
        return None